In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

In [ ]:
plots = Path("plots") / "compression"

In [ ]:
table = pd.read_csv("table-raw.csv")

In [ ]:
def compression_ratio_pick_single(var: str, percentile: str, pick: None | str = None) -> float:
    t = table[(table["level"] == "single") & (table["var"] == var) & (table["percentile"] == percentile)]
    pick = t["pick"].iloc[0] if pick is None else pick
    if pick == "absolute":
        return float(t["crlinquant"].iloc[0])
    if pick == "relative":
        return float(t["crbitround"].iloc[0])
    if pick == "quadratic":
        return float(t["crlinquantquadstep"].iloc[0])

In [ ]:
def compression_ratio_pick_pressure(var: str, level: str, percentile: str, pick: None | str = None) -> float:
    t = table[(table["level"] == level) & (table["var"] == var) & (table["percentile"] == percentile)]
    pick = t["pick"].iloc[0] if pick is None else pick
    if pick == "absolute":
        return float(t["crlinquant"].iloc[0])
    if pick == "relative":
        return float(t["crbitround"].iloc[0])
    if pick == "quadratic":
        return float(t["crlinquantquadstep"].iloc[0])

In [ ]:
import matplotlib
from matplotlib import pyplot as plt

In [ ]:
tab20c = matplotlib.colormaps["tab20c"].colors

In [ ]:
percentiles = ["100%", "99.9%", "99%", "95%"]
variables = sorted(table[table["level"] == "single"]["var"].unique()[:10])

x = np.arange(len(variables))
width = 0.25

fig, ax = plt.subplots(layout="constrained")

abs_handles = []
rel_handles = []
qua_handles = []

multiplier = 0
for i, p in enumerate(percentiles):
    offset = width * multiplier

    cr_abs = [compression_ratio_pick_single(var, p, pick="absolute") for var in variables]
    cr_rel = [compression_ratio_pick_single(var, p, pick="relative") for var in variables]
    cr_qua = [compression_ratio_pick_single(var, p, pick="quadratic") for var in variables]

    cr_all = np.array([[0.0, a, r, q, max(a, r, q) * 10] for a, r, q in zip(cr_abs, cr_rel, cr_qua)])
    cr_sor = np.sort(cr_all, axis=1)
    cr_idx = np.argsort(cr_all, axis=1)
    
    rects = ax.bar(
        x + offset,
        [csor[i] - csor[i-1] for csor, i in zip(cr_sor, cr_idx[:,1])],
        width,
        label=f"absolute {p}",
        bottom=[csor[i-1] for csor, i in zip(cr_sor, cr_idx[:,1])],
        color=tab20c[0+i],
    )
    abs_handles.append(rects)
    ax.bar_label(
        rects,
        labels=[f"a:{np.round(csor[i], 1)}" if i == (cr_idx.shape[1] - 2) else "" for csor, i in zip(cr_sor, cr_idx[:,1])],
        padding=3,
        rotation="vertical",
        color=tab20c[0],
    )

    rects = ax.bar(
        x + offset,
        [csor[i] - csor[i-1] for csor, i in zip(cr_sor, cr_idx[:,2])],
        width,
        label=f"relative {p}",
        bottom=[csor[i-1] for csor, i in zip(cr_sor, cr_idx[:,2])],
        color=tab20c[4+i],
    )
    rel_handles.append(rects)
    ax.bar_label(
        rects,
        labels=[f"r:{np.round(csor[i], 1)}" if i == (cr_idx.shape[1] - 2) else "" for csor, i in zip(cr_sor, cr_idx[:,2])],
        padding=3,
        rotation="vertical",
        color=tab20c[4],
    )

    rects = ax.bar(
        x + offset,
        [csor[i] - csor[i-1] for csor, i in zip(cr_sor, cr_idx[:,3])],
        width,
        label=f"quadratic {p}",
        bottom=[csor[i-1] for csor, i in zip(cr_sor, cr_idx[:,3])],
        color=tab20c[8+i],
    )
    qua_handles.append(rects)
    ax.bar_label(
        rects,
        labels=[f"q:{np.round(csor[i], 1)}" if i == (cr_idx.shape[1] - 2) else "" for csor, i in zip(cr_sor, cr_idx[:,3])],
        padding=3,
        rotation="vertical",
        color=tab20c[8],
    )
    
    multiplier += 1

ax.set_ylim(0.0, ax.set_ylim()[1] * 1.1)
ax.set_ylabel("Compression Ratio")
ax.set_title("Impact of Error Bounds on Compression Ratios of single-level variables")
ax.set_xticks(x + width, variables)
ax.legend(handles=abs_handles+rel_handles+qua_handles, ncol=3)

plt.show()
fig.savefig(plots / "single.pdf")

In [ ]:
percentiles = ["100%", "99.9%", "99%", "95%"]
variables = sorted(table[table["level"] != "single"]["var"].unique())

x = np.arange(len(variables))
width = 0.25

for level in ["1000.0 hPa", "850.0 hPa", "500.0 hPa", "50.0 hPa"]:
    fig, ax = plt.subplots(figsize=(10, 5), layout="constrained")
    
    abs_handles = []
    rel_handles = []
    qua_handles = []
    
    multiplier = 0
    for i, p in enumerate(percentiles):
        offset = width * multiplier
    
        cr_abs = [compression_ratio_pick_pressure(var, level, p, pick="absolute") for var in variables]
        cr_rel = [compression_ratio_pick_pressure(var, level, p, pick="relative") for var in variables]
        cr_qua = [compression_ratio_pick_pressure(var, level, p, pick="quadratic") for var in variables]
    
        cr_all = np.array([[0.0, a, r, q, max(a, r, q) * 10] for a, r, q in zip(cr_abs, cr_rel, cr_qua)])
        cr_sor = np.sort(cr_all, axis=1)
        cr_idx = np.argsort(cr_all, axis=1)
        
        rects = ax.bar(
            x + offset,
            [csor[i] - csor[i-1] for csor, i in zip(cr_sor, cr_idx[:,1])],
            width,
            label=f"absolute {p}",
            bottom=[csor[i-1] for csor, i in zip(cr_sor, cr_idx[:,1])],
            color=tab20c[0+i],
        )
        abs_handles.append(rects)
        ax.bar_label(
            rects,
            labels=[f"a:{np.round(csor[i], 1)}" if i == (cr_idx.shape[1] - 2) else "" for csor, i in zip(cr_sor, cr_idx[:,1])],
            padding=3,
            rotation="vertical",
            color=tab20c[0],
        )
    
        rects = ax.bar(
            x + offset,
            [csor[i] - csor[i-1] for csor, i in zip(cr_sor, cr_idx[:,2])],
            width,
            label=f"relative {p}",
            bottom=[csor[i-1] for csor, i in zip(cr_sor, cr_idx[:,2])],
            color=tab20c[4+i],
        )
        rel_handles.append(rects)
        ax.bar_label(
            rects,
            labels=[f"r:{np.round(csor[i], 1)}" if i == (cr_idx.shape[1] - 2) else "" for csor, i in zip(cr_sor, cr_idx[:,2])],
            padding=3,
            rotation="vertical",
            color=tab20c[4],
        )
    
        rects = ax.bar(
            x + offset,
            [csor[i] - csor[i-1] for csor, i in zip(cr_sor, cr_idx[:,3])],
            width,
            label=f"quadratic {p}",
            bottom=[csor[i-1] for csor, i in zip(cr_sor, cr_idx[:,3])],
            color=tab20c[8+i],
        )
        qua_handles.append(rects)
        ax.bar_label(
            rects,
            labels=[f"q:{np.round(csor[i], 1)}" if i == (cr_idx.shape[1] - 2) else "" for csor, i in zip(cr_sor, cr_idx[:,3])],
            padding=3,
            rotation="vertical",
            color=tab20c[8],
        )
        
        multiplier += 1

    if level in ["1000.0 hPa", "850.0 hPa"]:
        ax.set_ylim(ax.set_ylim()[0], ax.set_ylim()[1] * 1.1)
    else:
        ax.set_yscale("log")
        ax.set_ylim(ax.set_ylim()[0], ax.set_ylim()[1] * (3.0 if level == "500.0 hPa" else 5.0))
    
    ax.set_ylabel("Compression Ratio")
    ax.set_title(f"Impact of Error Bounds on Compression Ratios of pressure-level variables at {level}")
    ax.set_xticks(x + width, variables)
    ax.legend(handles=abs_handles+rel_handles+qua_handles, ncol=3)
    
    plt.show()
    fig.savefig(plots / f"pressure-{level.replace(' ', '')}.pdf")